# 👤 YOLOv8 Face Detection Training on Kaggle

**Setup Requirements:**
1. ✅ GPU: Tesla T4 (Settings → Accelerator → GPU T4)
2. ✅ Internet: ON
3. ✅ Dataset: WIDER FACE (already added)

**Expected Results:**
- Training Time: ~2 hours
- mAP@0.5: > 0.85
- Model Size: ~6 MB

## 📦 Step 1: Clone Repository & Install Dependencies

In [ ]:
# Clone GitHub repository
!git clone https://github.com/rohitKT-23/FaceDetection.git
%cd FaceDetection

# Install dependencies
!pip install ultralytics opencv-python albumentations matplotlib pillow pyyaml tqdm pandas -q

print("\n" + "="*60)
print("✓ Setup Complete!")
print("="*60)

## 🎮 Step 2: Verify GPU

In [ ]:
import torch

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print("\n✓ GPU Ready for Training!")
else:
    print("\n⚠️ GPU not enabled! Check Settings → Accelerator → GPU T4")

## 📂 Step 3: Copy Dataset Images

In [ ]:
import os

# Create directories
os.makedirs('data/widerface/images/train', exist_ok=True)
os.makedirs('data/widerface/images/val', exist_ok=True)
os.makedirs('data/widerface/labels/train', exist_ok=True)
os.makedirs('data/widerface/labels/val', exist_ok=True)

print("Copying training images...")
!cp -r /kaggle/input/facedetectiondataset/WIDER_train/WIDER_train/images/* data/widerface/images/train/

print("Copying validation images...")
!cp -r /kaggle/input/facedetectiondataset/WIDER_val/WIDER_val/images/* data/widerface/images/val/

print("\n✓ Images copied!")
!echo "Total train folders:"
!ls data/widerface/images/train/ | wc -l

## 🔄 Step 4: Convert Annotations to YOLO Format

In [ ]:
import os
from pathlib import Path
import cv2

def convert_wider_to_yolo(ann_file, img_dir, label_dir):
    """Convert WIDER FACE annotations to YOLO format"""
    os.makedirs(label_dir, exist_ok=True)
    
    with open(ann_file, 'r') as f:
        lines = f.readlines()
    
    i = 0
    converted = 0
    
    while i < len(lines):
        img_path = lines[i].strip()
        i += 1
        
        if i >= len(lines):
            break
        
        try:
            num_faces = int(lines[i].strip())
        except:
            i += 1
            continue
        
        i += 1
        
        img_file = os.path.join(img_dir, img_path)
        
        if not os.path.exists(img_file):
            i += num_faces
            continue
        
        img = cv2.imread(img_file)
        if img is None:
            i += num_faces
            continue
            
        img_h, img_w = img.shape[:2]
        
        boxes = []
        for _ in range(num_faces):
            if i >= len(lines):
                break
            
            parts = lines[i].strip().split()
            if len(parts) >= 4:
                x, y, w, h = map(int, parts[:4])
                
                if w > 0 and h > 0:
                    x_center = (x + w/2) / img_w
                    y_center = (y + h/2) / img_h
                    width = w / img_w
                    height = h / img_h
                    
                    boxes.append(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
            
            i += 1
        
        if boxes:
            label_file = os.path.join(label_dir, Path(img_path).stem + '.txt')
            os.makedirs(os.path.dirname(label_file), exist_ok=True)
            
            with open(label_file, 'w') as f:
                f.write('\n'.join(boxes))
            
            converted += 1
        
        if converted % 1000 == 0:
            print(f"Converted {converted} images...")
    
    print(f"✓ Conversion complete! Converted {converted} images")
    return converted

# Annotation file paths
train_ann = '/kaggle/input/facedetectiondataset/wider_face_split/wider_face_split/wider_face_train_bbx_gt.txt'
val_ann = '/kaggle/input/facedetectiondataset/wider_face_split/wider_face_split/wider_face_val_bbx_gt.txt'

# Convert train set
print("Converting training set...")
train_converted = convert_wider_to_yolo(train_ann, 'data/widerface/images/train', 'data/widerface/labels/train')

# Convert val set
print("\nConverting validation set...")
val_converted = convert_wider_to_yolo(val_ann, 'data/widerface/images/val', 'data/widerface/labels/val')

print(f"\n{'='*60}")
print(f"Train: {train_converted} images | Val: {val_converted} images")
print(f"{'='*60}")

## 🗂️ Step 5: Flatten Image Folders (IMPORTANT!)

In [ ]:
import shutil
from pathlib import Path

def flatten_images(img_dir):
    """Move all images from nested folders to parent folder"""
    img_dir = Path(img_dir)
    
    image_extensions = ['.jpg', '.jpeg', '.png']
    moved = 0
    
    for ext in image_extensions:
        for img_file in img_dir.rglob(f'*{ext}'):
            if img_file.parent != img_dir:
                dest = img_dir / img_file.name
                if dest.exists():
                    dest = img_dir / f"{img_file.stem}_{img_file.parent.name}{img_file.suffix}"
                shutil.move(str(img_file), str(dest))
                moved += 1
    
    # Remove empty folders
    for folder in img_dir.iterdir():
        if folder.is_dir():
            try:
                folder.rmdir()
            except:
                pass
    
    print(f"✓ Moved {moved} images")
    return moved

# Flatten train images
print("Flattening train images...")
train_moved = flatten_images('data/widerface/images/train')

# Flatten val images  
print("\nFlattening val images...")
val_moved = flatten_images('data/widerface/images/val')

# Verify
print(f"\n{'='*60}")
print("Verification:")
print(f"{'='*60}")
!find data/widerface/images/train -name "*.jpg" -type f | wc -l
!find data/widerface/labels/train -name "*.txt" -type f | wc -l

## ⚙️ Step 6: Update Training Config

In [ ]:
import yaml

with open('configs/yolov8_face.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Update for Kaggle GPU (Tesla T4 - 15GB memory)
config['path'] = '/kaggle/working/FaceDetection/data/widerface'
config['epochs'] = 50
config['batch'] = 16  # Reduced from 32 to avoid OOM errors
config['device'] = 0
config['workers'] = 8

with open('configs/yolov8_face.yaml', 'w') as f:
    yaml.dump(config, f)

print("✓ Config updated!")
print(f"  Path: {config['path']}")
print(f"  Epochs: {config['epochs']}")
print(f"  Batch: {config['batch']}")
print(f"  Device: GPU {config['device']}")

## 🚀 Step 7: Start Training (1-2 hours)

**Note:** This will take approximately 1-2 hours on Tesla T4 GPU

In [ ]:
# Start training
!python src/train.py --config configs/yolov8_face.yaml --model-size n

print("\n" + "="*60)
print("✓ Training Complete!")
print("="*60)

## 📊 Step 8: View Training Results

In [ ]:
import pandas as pd
from IPython.display import Image, display

# Load results
results = pd.read_csv('runs/detect/yolov8_face/results.csv')

print("Training Progress (Last 10 Epochs):")
print(results.tail(10))

# Final metrics
final = results.iloc[-1]
print(f"\n{'='*60}")
print("Final Metrics:")
print(f"{'='*60}")
print(f"  mAP@0.5: {final['metrics/mAP50(B)']:.4f}")
print(f"  mAP@0.5:0.95: {final['metrics/mAP50-95(B)']:.4f}")
print(f"  Precision: {final['metrics/precision(B)']:.4f}")
print(f"  Recall: {final['metrics/recall(B)']:.4f}")

# Display training plots
print("\nTraining Results:")
display(Image('runs/detect/yolov8_face/results.png'))

## 💾 Step 9: Save Model for Download

In [ ]:
# Copy files to Kaggle output directory
!cp runs/detect/yolov8_face/weights/best.pt /kaggle/working/best_yolov8_face.pt
!cp runs/detect/yolov8_face/weights/last.pt /kaggle/working/last_yolov8_face.pt
!cp runs/detect/yolov8_face/results.csv /kaggle/working/training_results.csv
!cp runs/detect/yolov8_face/results.png /kaggle/working/training_plots.png
!cp runs/detect/yolov8_face/confusion_matrix.png /kaggle/working/confusion_matrix.png 2>/dev/null || true

print("✓ Files saved to /kaggle/working/")
print("\nDownload from: Notebook → Output → Download")
print("\nSaved files:")
!ls -lh /kaggle/working/*.pt /kaggle/working/*.csv /kaggle/working/*.png 2>/dev/null

## 🎯 Step 10: Test Inference (Optional)

In [ ]:
# Test on validation images
!python src/infer.py \
    --model runs/detect/yolov8_face/weights/best.pt \
    --source data/widerface/images/val/ \
    --output-dir outputs/test \
    --conf 0.25 \
    --max-det 100

# Display sample results
import os
from IPython.display import Image, display

test_images = [f for f in os.listdir('outputs/test') if f.endswith(('.jpg', '.png'))]
if test_images:
    print(f"\nShowing {min(3, len(test_images))} sample detections:")
    for img in test_images[:3]:
        print(f"\n{img}:")
        display(Image(f'outputs/test/{img}'))
else:
    print("No test images found")



---

## 🎉 Training Complete!

### 📥 Download Your Model:
1. Go to **Notebook → Output** (top right)
2. Download `best_yolov8_face.pt`
3. Download `training_results.csv` for metrics

### 🚀 Next Steps:
1. Use the model locally with `src/infer.py`
2. Deploy with Streamlit: `streamlit run app/streamlit_app.py`
3. Export to ONNX for production (if using Python 3.10-3.12)

### 📊 Expected Performance:
- **mAP@0.5**: > 0.85
- **FPS on GPU**: 60+ FPS
- **Model Size**: ~6 MB

### 💡 Resume Bullet Point:
*"Developed a real-time face detection system using YOLOv8, achieving mAP@0.5 > 0.85 on WIDER FACE dataset with 60+ FPS on GPU, optimized for production deployment."*